In [40]:
import pandas as pd
import numpy as np

new_orders = pd.read_csv("../data_cleaned/new_orders_cleaned.csv")
shipments = pd.read_csv("../data_cleaned/shipments_cleaned.csv")
unfilled_orders = pd.read_csv("../data_cleaned/unfilled_orders_cleaned.csv")
total_inventories = pd.read_csv("../data_cleaned/total_inventories_cleaned.csv")
inventories_to_shipments = pd.read_csv("../data_cleaned/inventories_to_shipments_cleaned.csv")
unfilled_orders_to_shipments = pd.read_csv("../data_cleaned/unfilled_orders_to_shipments_cleaned.csv")

print(new_orders.columns)
print(shipments.columns)

Index(['series_code', 'date', 'year', 'month_num', 'new_orders'], dtype='str')
Index(['series_code', 'date', 'year', 'month_num', 'shipments'], dtype='str')


In [41]:
def create_base_code(code):
    code = str(code)

    suffixes = [
        "INVP",
        "VSP",
        "USP",
        "NOP",
        "MNO",
        "VS",
        "UO",
        "UP",
        "TI",
        "NO",
        "SP"
    ]

    for suffix in suffixes:
        if code.endswith(suffix):
            return code[:-len(suffix)]

    return code


datasets = [
    new_orders,
    shipments,
    unfilled_orders,
    total_inventories,
    inventories_to_shipments,
    unfilled_orders_to_shipments
]

for data in datasets:
    data["base_code"] = data["series_code"].apply(create_base_code)
    data["date"] = pd.to_datetime(data["date"])

print(new_orders[["series_code", "base_code"]].head())
print(shipments[["series_code", "base_code"]].head())
print(total_inventories[["series_code", "base_code"]].head())
print(unfilled_orders[["series_code", "base_code"]].head())

  series_code base_code
0      AMTMNO       AMT
1      AMTMNO       AMT
2      AMTMNO       AMT
3      AMTMNO       AMT
4      AMTMNO       AMT
  series_code base_code
0      AMTMVS      AMTM
1      AMTMVS      AMTM
2      AMTMVS      AMTM
3      AMTMVS      AMTM
4      AMTMVS      AMTM
  series_code base_code
0      AMTMTI      AMTM
1      AMTMTI      AMTM
2      AMTMTI      AMTM
3      AMTMTI      AMTM
4      AMTMTI      AMTM
  series_code base_code
0      AMTMUO      AMTM
1      AMTMUO      AMTM
2      AMTMUO      AMTM
3      AMTMUO      AMTM
4      AMTMUO      AMTM


In [42]:
new_orders_small = new_orders[
    ["base_code", "date", "year", "month_num", "new_orders"]
]

shipments_small = shipments[
    ["base_code", "date", "year", "month_num", "shipments"]
]

unfilled_orders_small = unfilled_orders[
    ["base_code", "date", "year", "month_num", "unfilled_orders"]
]

total_inventories_small = total_inventories[
    ["base_code", "date", "year", "month_num", "total_inventories"]
]

inventories_to_shipments_small = inventories_to_shipments[
    ["base_code", "date", "year", "month_num", "inventories_to_shipments"]
]

unfilled_orders_to_shipments_small = unfilled_orders_to_shipments[
    ["base_code", "date", "year", "month_num", "unfilled_orders_to_shipments"]
]

master = new_orders_small.merge(
    shipments_small,
    on=["base_code", "date", "year", "month_num"],
    how="outer"
)

master = master.merge(
    unfilled_orders_small,
    on=["base_code", "date", "year", "month_num"],
    how="outer"
)

master = master.merge(
    total_inventories_small,
    on=["base_code", "date", "year", "month_num"],
    how="outer"
)

master = master.merge(
    inventories_to_shipments_small,
    on=["base_code", "date", "year", "month_num"],
    how="outer"
)

master = master.merge(
    unfilled_orders_to_shipments_small,
    on=["base_code", "date", "year", "month_num"],
    how="outer"
)

display(master.head())
print(master.shape)
print(master.isna().sum())

,base_code,date,year,month_num,new_orders,shipments,unfilled_orders,total_inventories,inventories_to_shipments,unfilled_orders_to_shipments
0,A11A,1992-01-01,1992,1,NaN,3459.0,NaN,3375.0,NaN,NaN
1,A11A,1992-02-01,1992,2,NaN,3532.0,NaN,3345.0,NaN,NaN
2,A11A,1992-03-01,1992,3,NaN,3373.0,NaN,3345.0,NaN,NaN
3,A11A,1992-04-01,1992,4,NaN,3623.0,NaN,3291.0,NaN,NaN
4,A11A,1992-05-01,1992,5,NaN,3808.0,NaN,3349.0,NaN,NaN


(165096, 10)
base_code                            0
date                                 0
year                                 0
month_num                            0
new_orders                      122454
shipments                        93966
unfilled_orders                 123996
total_inventories                35220
inventories_to_shipments        145368
unfilled_orders_to_shipments    157698
dtype: int64


In [43]:
master_complete = master.dropna(
    subset=["new_orders", "shipments", "total_inventories", "unfilled_orders"]
)

print("Rows with all main values:", master_complete.shape[0])

display(master_complete[
    ["base_code", "date", "new_orders", "shipments", "total_inventories", "unfilled_orders"]
].head(20))

Rows with all main values: 37720


,base_code,date,new_orders,shipments,total_inventories,unfilled_orders
31921,A31A,1992-02-01,4449.0,4621.0,8236.0,7906.0
31922,A31A,1992-03-01,4880.0,4755.0,9803.0,8031.0
31923,A31A,1992-04-01,4815.0,4796.0,9799.0,8050.0
31924,A31A,1992-05-01,4960.0,4855.0,9741.0,8155.0
31925,A31A,1992-06-01,4976.0,4799.0,9767.0,8332.0
31926,A31A,1992-07-01,4928.0,4698.0,9704.0,8562.0
31927,A31A,1992-08-01,4748.0,4645.0,9738.0,8665.0
31928,A31A,1992-09-01,4503.0,4567.0,9734.0,8601.0
31929,A31A,1992-10-01,4327.0,4524.0,9654.0,8404.0
31930,A31A,1992-11-01,4411.0,4562.0,9587.0,8253.0


In [44]:
master = master.sort_values(["base_code", "date"])

master["new_orders_mom_growth"] = master.groupby("base_code")["new_orders"].pct_change()
master["shipments_mom_growth"] = master.groupby("base_code")["shipments"].pct_change()
master["unfilled_orders_mom_growth"] = master.groupby("base_code")["unfilled_orders"].pct_change()
master["inventories_mom_growth"] = master.groupby("base_code")["total_inventories"].pct_change()

master["orders_to_shipments_ratio"] = master["new_orders"] / master["shipments"]
master["inventory_to_shipments_ratio_calc"] = master["total_inventories"] / master["shipments"]

display(master.head())

,base_code,date,year,month_num,new_orders,shipments,unfilled_orders,total_inventories,inventories_to_shipments,unfilled_orders_to_shipments,new_orders_mom_growth,shipments_mom_growth,unfilled_orders_mom_growth,inventories_mom_growth,orders_to_shipments_ratio,inventory_to_shipments_ratio_calc
0,A11A,1992-01-01,1992,1,NaN,3459.0,NaN,3375.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.975716
1,A11A,1992-02-01,1992,2,NaN,3532.0,NaN,3345.0,NaN,NaN,NaN,0.021104,NaN,-0.008889,NaN,0.947055
2,A11A,1992-03-01,1992,3,NaN,3373.0,NaN,3345.0,NaN,NaN,NaN,-0.045017,NaN,0.000000,NaN,0.991699
3,A11A,1992-04-01,1992,4,NaN,3623.0,NaN,3291.0,NaN,NaN,NaN,0.074118,NaN,-0.016143,NaN,0.908363
4,A11A,1992-05-01,1992,5,NaN,3808.0,NaN,3349.0,NaN,NaN,NaN,0.051063,NaN,0.017624,NaN,0.879464


In [45]:
master.to_csv("../data_cleaned/m3_master_with_kpis.csv", index=False)

print("Corrected KPI master dataset saved.")

Corrected KPI master dataset saved.


In [46]:
master_complete = master.dropna(
    subset=["new_orders", "shipments", "total_inventories", "unfilled_orders"]
)

print("Rows with all main values:", master_complete.shape[0])

display(master_complete[
    ["base_code", "date", "new_orders", "shipments", "total_inventories", "unfilled_orders"]
].head(20))

Rows with all main values: 37720


,base_code,date,new_orders,shipments,total_inventories,unfilled_orders
31921,A31A,1992-02-01,4449.0,4621.0,8236.0,7906.0
31922,A31A,1992-03-01,4880.0,4755.0,9803.0,8031.0
31923,A31A,1992-04-01,4815.0,4796.0,9799.0,8050.0
31924,A31A,1992-05-01,4960.0,4855.0,9741.0,8155.0
31925,A31A,1992-06-01,4976.0,4799.0,9767.0,8332.0
31926,A31A,1992-07-01,4928.0,4698.0,9704.0,8562.0
31927,A31A,1992-08-01,4748.0,4645.0,9738.0,8665.0
31928,A31A,1992-09-01,4503.0,4567.0,9734.0,8601.0
31929,A31A,1992-10-01,4327.0,4524.0,9654.0,8404.0
31930,A31A,1992-11-01,4411.0,4562.0,9587.0,8253.0
